# Datamine COPYMOD Process Example

This notebook demonstrates how to configure and run the **`copymod`** process wrapper in `dmstudio`.

## Process Description

## COPYMOD

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

**Note** : This process supports **[retrieval criteria](<../COMMON/Retrieval_Criteria_Overview.md>)**.

Copy a normal or rotated model to a rotated or normal model with different origin and/or rotations.

The **COPYMOD** process creates a copy of a model using both translation and rotation. This can be summarised as one of four types using the @**MODTYPE** parameters:

MODTYPE | Input Model | Output Model | Comment
---|---|---|---
1 | Normal | Normal | Translation only
2 | Normal | Rotated | Apply translation and rotation
3 | Rotated | Normal | Remove rotated
4 | Rotated | Rotated | Apply translation and rotation

#### Rotated Models

A rotated model file includes information on two grids the world (normal) grid and the local (rotated) grid. The normal origin, [**XYZ**]0, and rotated origin, [**XYZ**]**MORIG** , fields together with the rotation angles, **ANGLE**[**123**], and axes, **ROTAXIS**[**123**], provide the relationship between the two grids.

The **XC** , **YC** and **ZC** fields in a rotated model file hold the cell centre coordinates for the local (rotated) grid. It is sometimes helpful to see the corresponding coordinates in the world (normal) grid. When using **MODTYPE** s 2 and 4 the **COPYMOD** process allows the world coordinates to be added to the output model. This is done by specifying names for the coordinate fields - * **XWORLD** , * **YWORLD** and * **ZWORLD**.

In order to add world coordinates to an existing model use **MODTYPE** 4 but do not select any of the nine parameters **[XYZ**]**NEWORIG** , **AXIS**[**123**] and **ROTAXIS**[**123**], but do specify field names for the three [**XYZ**]**WORLD** fields. The output model will then be an exact copy of the input model but with the addition of the three world coordinate fields.

### Input Files:

* **modelin** (Block Model File):
  Input model prototype This is a standard block model file containing the 13 compulsory
  fields. It may also contain the rotated model fields.
  Required=Yes

### Output Files:

* **modelout** (Block Model File):
  Output model containing estimated MIK grades, etc.
  Required=Yes

### Fields:

* **xworld** (Numeric : IN):
  X coordinate of sample data in **SAMPLES** file. If not specified, then XPT is assumed.
  Default=Undefined
  Required=No

* **yworld** (Numeric : IN):
  Y coordinate of sample data in **SAMPLES** file. If not specified, then YPT is assumed.
  Default=Undefined
  Required=No

* **zworld** (Numeric : IN):
  Z coordinate of sample data in **SAMPLES** file. If not specified, then ZPT is assumed.
  Default=Undefined
  Required=No

### Parameters:

* **modtype**:
  Output Model Type
  Range=
  Values=
  Default=
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('copymod')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute copymod
print("Running copymod...")
dm_cmd.copymod(
    modelin_i='t_mod1',  # required
    modelout_o='t_copymod_out',  # required
    # xworld_f='optional',  # optional
    # yworld_f='optional',  # optional
    # zworld_f='optional',  # optional
    # modtype_p='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("copymod execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_copymod_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")